# 4-SFT

对应 `trainer/train_full_sft.py`。SFT 让模型适应 `user / assistant / system / tool` 模板，同时继续灌知识和行为。

MiniMind-3 的 `sft_t2t(_mini).jsonl` **已经混入 Tool Call 和思考样本**，默认 `full_sft` 就有基础 tool call 能力，通常不必再单独训一轮。


In [ ]:
import os, sys, math, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
import torch
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("../model")
print("device:", device, "vocab:", tokenizer.vocab_size)
def tiny_model(use_moe=False):
    cfg = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=use_moe)
    return MiniMindForCausalLM(cfg).to(device), cfg


## 看一条 SFT 样本如何被模板展开


In [ ]:
ds = SFTDataset("./toydata/sft_data.jsonl", tokenizer, max_length=160)
input_ids, labels = ds[0]
print(tokenizer.decode(input_ids, skip_special_tokens=False)[:500])
print("supervised / total:", int((labels != -100).sum()), "/", len(labels))


Tool Call 样本（最后一条 toy 数据）：


In [ ]:
input_ids, labels = ds[-1]
print(tokenizer.decode(input_ids, skip_special_tokens=False))


## 一步训练

和预训练同一套 `model(..., labels=)`，只是 Dataset 换了、loss 只打在 assistant 段。


In [ ]:
model, cfg = tiny_model()
loader = DataLoader(ds, batch_size=2, shuffle=True)
optimizer = optim.AdamW(model.parameters(), lr=5e-4)
model.train()
for step, (input_ids, labels) in enumerate(loader, start=1):
    input_ids, labels = input_ids.to(device), labels.to(device)
    out = model(input_ids, labels=labels)
    loss = out.loss + out.aux_loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(f"step {step} loss={float(loss):.4f}")


完整训练：`cd trainer && python train_full_sft.py`。测对话：`python eval_llm.py --weight full_sft`。测工具：`python scripts/eval_toolcall.py --weight full_sft`。
